## 第 2 课：1-D Grid 与 Mask（双输入）

题目：[Triton: Vector Addition Kernel](https://www.deep-ml.com/problems/968?from=Triton%20Essentials)（ID 968）

计算目标：

In [ ]:
output[i] = x[i] + y[i]

x、y、output 都是长度为 `n` 的一维 Tensor。

例如：

In [ ]:
x = [1, 2, 3, 4]
y = [10, 20, 30, 40]

output = [11, 22, 33, 44]

这是最标准的 **load → compute → store** 三明治：两个输入、一个输出，逐元素运算。它是后面所有 kernel 的模板。

### 1. 与 Fill 的区别：多了一个输入 load

第 1 课只有 store、没有 load。这一课多了两步：

In [ ]:
x_block = tl.load(x_ptr + offsets, mask=mask)
y_block = tl.load(y_ptr + offsets, mask=mask)
result = x_block + y_block

offsets 和 mask 的构造与 Fill 完全一样：

In [ ]:
pid = tl.program_id(0)
offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
mask = offsets < n_elements

### 2. Mask 同时保护读和写

- `tl.load(ptr + offsets, mask=mask)`：mask=False 的 lane **不会真正访问内存**（返回 `other`，默认 0.0），防止读到越界地址的垃圾值。
- `tl.store(ptr + offsets, value, mask=mask)`：mask=False 的 lane **不写**，防止把数组末尾之外的内存写坏。

### 3. tl.constexpr 与 cdiv

In [ ]:
BLOCK_SIZE: tl.constexpr

告诉 Triton 这是编译期常量：块大小参与循环展开和向量化，运行期不能变。grid 用：

In [ ]:
grid = (triton.cdiv(n_elements, BLOCK_SIZE),)

`cdiv(a, b) = ceil(a / b)`，保证 program 数量足够覆盖所有元素。

## 你的代码骨架

In [ ]:
import torch
import triton
import triton.language as tl


@triton.jit
def add_kernel(
    x_ptr,
    y_ptr,
    output_ptr,
    n_elements,
    BLOCK_SIZE: tl.constexpr,
):
    # TODO 1：取得 program ID

    # TODO 2：生成 offsets

    # TODO 3：生成 mask

    # TODO 4：加载 x

    # TODO 5：加载 y

    # TODO 6：相加

    # TODO 7：写回 output
    pass


def add(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    # TODO 8：检查 x、y 形状一致

    # TODO 9：分配 output（形状、dtype 与 x 相同）

    BLOCK_SIZE = 1024

    # TODO 10：创建一维 grid

    # TODO 11：启动 kernel

    # TODO 12：返回 output
    pass

同时回答：

1. `n = 5000, BLOCK_SIZE = 1024` 时，grid 是多少？最后一个 program 的 offsets 区间是什么？其中多少个有效元素？
2. 为什么 `tl.load` 和 `tl.store` 都要传 mask？如果只给 store 传 mask 会怎样？
3. 为什么 `BLOCK_SIZE` 必须标注为 `tl.constexpr`？

把代码和三个答案发给我，我继续审查。